# Actividad 6 – Embudo de conversión por dispositivo
### Iván López - A01284875

## Metodología
* Leer el archivo csv con PySpark.
* Obtener métricas de volumen por tipo de evento
* Construir el embudo por dispositivo
* Calcular tasa de conversión
* Interpretar resultados

## Librerías requeridas

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## Configuración de Spark

In [2]:
# Crear sesión Spark
try:
    sc.stop()
except Exception:
    pass

In [3]:
spark = (SparkSession.builder
         .appName("NotebookSession")
         .master("local[*]")
         .config("spark.ui.port", "0")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("WARN")

print(spark.version, sc.appName)

4.0.1 NotebookSession


## Carga de datos

In [4]:
# leer el CSV
df_cs = spark.read.csv('activity6_clickstream_big.csv', header=True, inferSchema=True) 

In [5]:
df_cs = (df_cs.withColumn('event_ts', F.to_timestamp('event_ts')))

In [6]:
# verificar esquema
df_cs.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- page: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- device: string (nullable = true)



In [7]:
#primeras filas
df_cs.show(10)

+--------------------+-------+-------------------+---------+-----------+-------+
|            event_id|user_id|           event_ts|     page| event_type| device|
+--------------------+-------+-------------------+---------+-----------+-------+
|9b75953d-9859-482...|  42600|2025-02-17 11:02:42|  /search|       view| mobile|
|45e8ea33-7700-409...|  49736|2025-02-14 22:02:15|/checkout|   checkout|desktop|
|a84fdcd2-63cf-405...|  45953|2025-02-01 14:13:30|  /search|       view| tablet|
|f981042c-9ed3-4bd...|  39634|2025-02-14 04:22:50|    /home|       view|desktop|
|2ae15c70-8726-4f3...|  46400|2025-02-13 19:59:25|    /cart|   purchase| mobile|
|2b9134af-8840-48a...|  15900|2025-03-01 16:30:04|/checkout|       view| tablet|
|6e45f524-c954-415...|  28318|2025-02-10 11:11:04| /product|   purchase|desktop|
|76db3170-02e3-440...|   7680|2025-02-11 09:40:39|    /home|add_to_cart| mobile|
|14683cf0-af22-456...|  43292|2025-02-15 14:37:59| /product|       view|desktop|
|16b60d9b-add4-4d0...|   790

## Volumen por tipo de evento

In [12]:
vol_eventType=(df_cs.groupBy('event_type')
                .agg(F.count('*').alias('volume')))

vol_eventType.show()

+-----------+------+
| event_type|volume|
+-----------+------+
|   purchase|  4106|
|add_to_cart|  7989|
|       view| 23950|
|   checkout|  3955|
+-----------+------+



Hay 23950 eventos del tipo view, 7989 del tipo add_to_cart y 4106 del tipo purchase, dando a entender que las personas no compran todo lo que les interesó y agregaron al carrito y mucho menos compran todo producto que ven.

## Embudo por dispositivo

In [19]:
embudo_disp=(df_cs.groupBy('device')
            .pivot('event_type')
            .agg(F.count(F.lit(1)))
            .na.fill(0))

embudo_disp.show()

+-------+-----------+--------+--------+----+
| device|add_to_cart|checkout|purchase|view|
+-------+-----------+--------+--------+----+
|desktop|       2634|    1344|    1341|7929|
| mobile|       2736|    1286|    1342|7939|
| tablet|       2619|    1325|    1423|8082|
+-------+-----------+--------+--------+----+



## Con tasa de conversión

In [23]:
disp_conv=(embudo_disp
            .withColumn('conv_view_to_purchase', F.round(F.col('purchase') / F.col('view'), 2))
            .orderBy(F.desc('conv_view_to_purchase')))

disp_conv.show()

+-------+-----------+--------+--------+----+---------------------+
| device|add_to_cart|checkout|purchase|view|conv_view_to_purchase|
+-------+-----------+--------+--------+----+---------------------+
| tablet|       2619|    1325|    1423|8082|                 0.18|
|desktop|       2634|    1344|    1341|7929|                 0.17|
| mobile|       2736|    1286|    1342|7939|                 0.17|
+-------+-----------+--------+--------+----+---------------------+



## Interpretación
El dispositivo con mejor desempeño es la Tablet, con una tasa de conversión de view a purchase del 18% (0.18). 

En cuanto a dónde se rompe el embudo, primeramente entre view y add-to-cart es que se pierde la mayor masa de usuarios en todos los dispositivos. Asimismo, hay un quiebre crítico en Mobile entre add-to-cart (más eventos de este tipo por dispositivo) y checkout (menos eventos de este tipo por dispositivo) ya que los usuarios llenan el carrito pero no finalizan el pago. Además, existe una anomalía técnica grave tanto en Tablet como Mobile donde se registran más compras que checkouts. 

Las hipótesis que se plantearían para mejorar la conversión de view a purchase consisten en: 
* Implementar pagos rápidos con billeteras digitales (como Apple Pay y Google Pay) para recuperar las ventas en Mobile que se pierden dado que el proceso de pago en el celular puede resultar demasiado largo o incómodo, causando que los usuarios abandonen el carrito.
* Auditar la medición de los datos actuales de checkout en Tablet y Mobile que son inconsistentes para verificar la implementación técnica de los eventos y asegurar que se está midiendo la conversión real y no se está perdiendo data de pasos intermedios.